# DataFrame API Cheatsheet

Quick reference for PySpark DataFrame operations.

## Table of Contents
- [Creating DataFrames](#creating-dataframes)
- [Basic Operations](#basic-operations)
- [Transformations](#transformations)
- [Actions](#actions)
- [Joins](#joins)
- [Window Functions](#window-functions)
- [Complex Types](#complex-types)

## Creating DataFrames

In [ ]:
# From list of tuples
data = [("Alice", 25), ("Bob", 30)]
df = spark.createDataFrame(data, ["name", "age"])

# From pandas DataFrame
import pandas as pd
pdf = pd.DataFrame({"name": ["Alice"], "age": [25]})
df = spark.createDataFrame(pdf)

# From RDD
rdd = sc.parallelize([("Alice", 25)])
df = spark.createDataFrame(rdd, ["name", "age"])

# Reading files
df = spark.read.csv("path/to/file.csv", header=True)
df = spark.read.json("path/to/file.json")
df = spark.read.parquet("path/to/file.parquet")

## Basic Operations

In [ ]:
# Display schema
df.printSchema()
df.schema

# Show data
df.show()
df.show(5, truncate=False)
df.head(5)
df.take(5)

# Column operations
df.select("name", "age")
df.select(df.name, df.age + 1)
df.withColumn("age_plus_one", df.age + 1)
df.withColumnRenamed("name", "full_name")
df.drop("age")

# Filtering
df.filter(df.age > 25)
df.where(df.age == 25)
df.filter((df.age > 25) & (df.age < 30))
df.filter(df.name.startswith("A"))

# Sorting
df.orderBy(df.age)
df.sort(df.age.desc())
df.orderBy(["age", "name"])

## Transformations

In [ ]:
from pyspark.sql.functions import col, lit, when, upper, lower, trim

# Add literal column
df.withColumn("country", lit("USA"))

# Conditional column
df.withColumn("age_group", when(df.age < 18, "minor").otherwise("adult"))

# String operations
df.withColumn("name_upper", upper(df.name))
df.withColumn("name_lower", lower(df.name))
df.withColumn("name_trimmed", trim(df.name))

# Numeric operations
df.withColumn("age_doubled", df.age * 2)
df.withColumn("age_squared", df.age ** 2)

# Distinct
df.distinct()
df.dropDuplicates()
df.dropDuplicates(["name"])

# Sampling
df.sample(0.1)  # 10% sample
df.sample(False, 0.1, 42)  # without replacement, 10%, seed 42

## Actions

In [ ]:
# Collect data
df.collect()  # Returns list of Row objects
df.collect()[0]["name"]  # Access first row, name column

# Count
df.count()  # Number of rows

# Write
df.write.csv("path/to/output.csv", header=True)
df.write.parquet("path/to/output.parquet")
df.write.json("path/to/output.json")
df.write.mode("overwrite").parquet("path/to/output.parquet")
df.write.mode("append").parquet("path/to/output.parquet")

# Cache
df.cache()  # Cache in memory
df.persist()  # Persist with default storage level
df.unpersist()  # Remove from cache

# Take/limit
df.limit(5).collect()
df.take(5)
df.first()

# Reduce
from pyspark.sql.functions import count, sum, avg, max, min
df.select(count("*")).collect()
df.select(sum(df.age)).collect()
df.select(avg(df.age)).collect()

## Aggregations

In [ ]:
from pyspark.sql.functions import count, sum, avg, max, min, stddev

# Simple aggregations
df.groupBy("department").count()
df.groupBy("department").sum("salary")
df.groupBy("department").avg("salary")
df.groupBy("department").max("salary")
df.groupBy("department").min("salary")

# Multiple aggregations
df.groupBy("department").agg(
    count("*").alias("count"),
    sum("salary").alias("total_salary"),
    avg("salary").alias("avg_salary")
)

# Multiple groupBy columns
df.groupBy(["department", "city"]).count()

## Joins

In [ ]:
# Inner join (default)
df1.join(df2, "id")
df1.join(df2, df1.id == df2.id)

# Left join
df1.join(df2, "id", "left")
df1.join(df2, df1.id == df2.id, "left")

# Right join
df1.join(df2, "id", "right")

# Outer join
df1.join(df2, "id", "outer")

# Left semi join (filter)
df1.join(df2, "id", "leftsemi")

# Left anti join (anti-join)
df1.join(df2, "id", "leftanti")

# Join on multiple columns
df1.join(df2, ["id", "department"])

# Join with alias
df1.alias("a").join(df2.alias("b"), col("a.id") == col("b.id"))

## Window Functions

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, rank, dense_rank, lag, lead

# Define window specification
windowSpec = Window.partitionBy("department").orderBy(df.salary.desc())

# Ranking functions
df.withColumn("row_num", row_number().over(windowSpec))
df.withColumn("rank", rank().over(windowSpec))
df.withColumn("dense_rank", dense_rank().over(windowSpec))

# Analytic functions
df.withColumn("lag_salary", lag("salary", 1).over(windowSpec))
df.withColumn("lead_salary", lead("salary", 1).over(windowSpec))

# Aggregate functions over window
from pyspark.sql.functions import sum, avg
df.withColumn("dept_total", sum("salary").over(Window.partitionBy("department")))
df.withColumn("dept_avg", avg("salary").over(Window.partitionBy("department")))

## Complex Types

In [ ]:
from pyspark.sql.functions import explode, array, struct, col

# Array operations
df.withColumn("items", array("item1", "item2"))
df.withColumn("exploded", explode(col("items")))

# Struct operations
df.withColumn("person", struct("name", "age"))
df.select(col("person.name"), col("person.age"))

# Map operations
from pyspark.sql.functions import create_map, map_keys, map_values
df.withColumn("mapping", create_map(col("key"), col("value")))
df.select(map_keys(col("mapping")))
df.select(map_values(col("mapping")))

## Performance Tips

- Use **DataFrame API** instead of RDD API when possible
- **cache()** frequently used DataFrames
- **repartition()** to increase parallelism (full shuffle)
- **coalesce()** to decrease partitions (no shuffle)
- Use **broadcast joins** for small tables
- **filter early** to reduce data volume
- Use **partitionBy** when writing for faster reads